In [ ]:
import string,re, nltk,os
from nltk.corpus import stopwords
from tensorflow.keras.preprocessing.text import Tokenizer


In [ ]:
#load the doc in memory
def load_doc(filename):
    f = open(filename)
    text =f.read()
    f.close()
    return text

In [ ]:
load_doc('C:/Users/UserH/Desktop/NLP/NLP_New/Deep Learning for NLP_Code_Day1/cv000_29590.txt')


In [ ]:
from nltk.tokenize import word_tokenize
swords = stopwords.words('english')

In [ ]:
#Function to clean the doc
def clean_doc(doc):
    tokens =word_tokenize(doc)
    tokens = [token for token in tokens if token.isalpha()]
    tokens = [token for token in tokens if token not in swords]
    tokens = [token for token in tokens if len(token)> 1]
    return tokens


In [ ]:
sample_data = load_doc('C:/Users/UserH/Desktop/NLP/NLP_New/Deep Learning for NLP_Code_Day1/cv000_29590.txt')
clean_doc(sample_data)

In [ ]:
f = open('C:/Users/UserH/Desktop/NLP/NLP_New/Deep Learning for NLP_Code_Day1/vocab.txt')
vocab = f.read().split()
vocab

In [ ]:
## load the file, clean the data and return lines

def doc_to_line(filename):
    doc = load_doc(filename)
    tokens = clean_doc(doc)
    tokens = [token for token in tokens if token in vocab]
    return' '. join(tokens)

In [ ]:
doc_to_line('C:/Users/UserH/Desktop/NLP/NLP_New/Deep Learning for NLP_Code_Day1/cv000_29590.txt')

In [ ]:
os.listdir('C:/Users/UserH/Desktop/NLP/NLP_New/Deep Learning for NLP_Code_Day1/txt_sentoken-20251128T094412Z-1-001/txt_sentoken/pos')

In [ ]:
# load all the files from directory

def process_train(directory):
    documents = []
    for filename in os.listdir(directory):
        if not filename.startswith('cv9'):
            path = directory +'/' + filename
            docs = load_doc(path)
            tokens = clean_doc(docs)
            documents.append(tokens)
    return documents

In [ ]:
tr = process_train('C:/Users/UserH/Desktop/NLP/NLP_New/Deep Learning for NLP_Code_Day1/txt_sentoken-20251128T094412Z-1-001/txt_sentoken/pos/')

In [ ]:
len(tr)

In [ ]:
# load all the files from directory

def process_test(directory):
    documents = []
    for filename in os.listdir(directory):
        if filename.startswith('cv9'):
            path = directory +'/' + filename
            docs = load_doc(path)
            tokens = clean_doc(docs)
            documents.append(tokens)
    return documents

In [ ]:
ts = process_test('C:/Users/UserH/Desktop/NLP/NLP_New/Deep Learning for NLP_Code_Day1/txt_sentoken-20251128T094412Z-1-001/txt_sentoken/pos/')

In [ ]:
len(ts)

In [ ]:
def process_docs(directory,is_train):
    documents = []
    for filename in os.listdir(directory):
        if is_train and filename.startswith('cv9'):
            continue
        if not is_train and not filename.startswith('cv9'):
            continue

        path = directory +'/' + filename
        docs = load_doc(path)
        tokens = clean_doc(docs)
        documents.append(tokens)
    return documents

In [ ]:
tr = process_docs('C:/Users/UserH/Desktop/NLP/NLP_New/Deep Learning for NLP_Code_Day1/txt_sentoken-20251128T094412Z-1-001/txt_sentoken/pos/',True)
len(tr)

In [ ]:
ts = process_docs('C:/Users/UserH/Desktop/NLP/NLP_New/Deep Learning for NLP_Code_Day1/txt_sentoken-20251128T094412Z-1-001/txt_sentoken/pos/',False)
len(ts)

In [ ]:
def load_data(is_train):
    neg = process_docs('C:/Users/UserH/Desktop/NLP/NLP_New/Deep Learning for NLP_Code_Day1/txt_sentoken-20251128T094412Z-1-001/txt_sentoken/neg//',is_train)
    pos = process_docs('C:/Users/UserH/Desktop/NLP/NLP_New/Deep Learning for NLP_Code_Day1/txt_sentoken-20251128T094412Z-1-001/txt_sentoken/pos//',is_train)
    docs = neg +pos
    labels = [0 for i in range ((len(neg)))] + [ 1 for i in range ((len(pos)))]
    return docs,labels


In [ ]:
## Load training data

train_data, train_labels = load_data(True)

# load testing data

test_data, test_labels = load_data(False)



In [ ]:
len(train_data), len (train_labels)

In [ ]:
len(test_data), len (test_labels)

In [ ]:

import numpy as np

np.unique(test_labels)

## Data Preparation

In [ ]:
def create_tokenizer (lines):
    tokenizer = Tokenizer()
    tokenizer.fit_on_texts(lines)
    return tokenizer

In [ ]:
tokenizer = create_tokenizer(train_data)

In [ ]:
##encode the training data
#x_train = tokenizer.texts_to_matrix(train_data)

In [ ]:
#x_train.shape

In [ ]:
from tensorflow.keras.preprocessing.sequence import pad_sequences

In [ ]:
def encode_docs(tokenizer, max_length, docs):
    encoded = tokenizer.texts_to_sequences(docs)
    padded = pad_sequences(encoded, maxlen = max_length, padding = 'post')
    return padded

In [ ]:
max_length = max([len(sent) for sent in train_data])

In [ ]:
max_length

In [ ]:
x_train = encode_docs(tokenizer, max_length, train_data)
x_test = encode_docs(tokenizer, max_length, test_data)

In [ ]:
x_train.shape
#x_test.shape

In [ ]:
len(tokenizer.word_index)

In [ ]:
##encode the testing data (no need )
# x_test = tokenizer.texts_to_matrix(test_data)
# x_test.shape


## ##Build the model

In [ ]:
from keras.models import Sequential
from keras.layers import Dense,Input, Conv1D, MaxPool1D, Flatten, Embedding
from keras.utils import plot_model

In [ ]:
vocab_size = len(vocab)

In [ ]:
print("vocab_size =", vocab_size)
print("max x_train index =", np.max(x_train))
print("x_train shape =", x_train.shape)

In [ ]:
vocab_size = len(tokenizer.word_index)+1
print("vocab_size =", vocab_size)

In [ ]:
# def define_model():
#     model = Sequential()
#     model.add(Embedding(vocab_size, 100))
#     model.add(Conv1D(filters = 16, kernel_size=5, activation = 'relu')) #filters=128
#     model.add(MaxPool1D(pool_size=2))
#     model.add(Flatten())
#     model.add(Dense(10, activation = 'relu'))
#     model.add(Dense(1, activation = 'sigmoid'))
#     model.compile(optimizer = 'adam', loss = 'binary_crossentropy', metrics = ['accuracy'])
#     return model

In [ ]:
# def define_model(vocab_size, max_length):
#     model = Sequential()
#     model.add(Embedding(input_dim=vocab_size, output_dim=100,input_length=max_length))
#     model.add(Conv1D(filters = 64, kernel_size=5, activation = 'relu')) #filters=128
#     model.add(MaxPool1D(pool_size=2))
#     model.add(Flatten())
#     model.add(Dense(32, activation = 'relu'))
#     model.add(Dense(1, activation = 'sigmoid'))
#     model.compile(optimizer = 'adam', loss = 'binary_crossentropy', metrics = ['accuracy'])
#     return model
#     return model

In [ ]:
def define_model(vocab_size):
    model = Sequential()
    model.add(Embedding(input_dim=vocab_size, output_dim=100))
    model.add(Conv1D(filters = 64, kernel_size=5, activation = 'relu')) #filters=128
    model.add(GlobalMaxPooling1D())
    model.add(Dense(32, activation = 'relu'))
    model.add(Dense(1, activation = 'sigmoid'))
    model.compile(optimizer = 'adam', loss = 'binary_crossentropy', metrics = ['accuracy'])
    return model

In [ ]:
#model = define_model()
from tensorflow.keras.layers import GlobalMaxPooling1D
model = define_model(vocab_size)

## Train the model

In [ ]:
model.fit(x_train,np.array(train_labels),epochs=10,batch_size=10)

In [ ]:
print(x_train.shape)
print(x_test.shape)

In [ ]:
model.evaluate(x_test,np.array(test_labels),batch_size=1)

In [ ]:
text1 = 'Best movie ever! It was great, I will definitely recommend it.'

text2 = 'This is the bad movie. Please dont watch it.'


In [ ]:
def predict_sentiment(review):
    tokens = clean_doc(review)
    line=' ' .join(tokens)
    # encoded = tokenizer.texts_to_matrix([tokens])
    encoded = encode_docs(tokenizer, max_length, line)
    yhat = model. predict(encoded, verbose=0)
    percent_pos = yhat[0,0]
    if round(percent_pos) == 0:
        return (1-percent_pos), 'Negative'
    return percent_pos,'Positive'

In [ ]:
predict_sentiment(text1)

In [ ]:
predict_sentiment(text2)

In [ ]:
clean_doc(text1)